# Step 1

Write the `BasicTokenizer` class, with the following three core functions:

- `def train(self, text, vocab_size, verbose=False)`
- `def encode(self, text)`
- `def decode(self, ids)`

Train your tokenizer on whatever text you like and visualize the merged tokens. Do they look reasonable? One default test you may wish to use is the text file `tests/taylorswift.txt`.


In [2]:
class BasicTokenizer:
    def __init__(self):
        self.lookup_table: dict[tuple[int, int], int] = dict()
    
    def train(self, text, vocab_size, verbose=False):
        tokens = list(map(int, text.encode("utf-8")))
        next_free_integer = 256
        while next_free_integer < vocab_size:
            occurrences = {}
            max_byte_occurrence = 0
            max_byte_tuple = None
            for i in range(len(tokens) - 1):
                token_tuple = (tokens[i], tokens[i + 1])
                occurrence_count = occurrences.get(token_tuple, 0) + 1
                occurrences[token_tuple] = occurrence_count
                if occurrence_count > max_byte_occurrence:
                    max_byte_occurrence = occurrence_count
                    max_byte_tuple = token_tuple
            if max_byte_occurrence < 2:
                return
            self.lookup_table[max_byte_tuple] = next_free_integer
            if verbose:
                print(f"Merging {max_byte_tuple} -> {next_free_integer} (occurred {max_byte_occurrence} times)")
            condensed_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == max_byte_tuple:
                    condensed_tokens.append(next_free_integer)
                    i += 2
                else:
                    condensed_tokens.append(tokens[i])
                    i += 1
            tokens = condensed_tokens
            next_free_integer += 1

    def encode(self, text) -> list[int]:
        tokens = list(map(int, text.encode("utf-8")))
        for pair, new_id in self.lookup_table.items():
            i = 0
            condensed_tokens = []
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                    condensed_tokens.append(new_id)
                    i += 2
                else:
                    condensed_tokens.append(tokens[i])
                    i += 1
            tokens = condensed_tokens
        return tokens

    def decode(self, tokens: list[int]) -> str:
        for pair, new_id in reversed(self.lookup_table.items()):
            decompressed_tokens = []
            for i in range(0, len(tokens)):
                if tokens[i] == new_id:
                    decompressed_tokens.append(pair[0])
                    decompressed_tokens.append(pair[1])
                else:
                    decompressed_tokens.append(tokens[i])
            tokens = decompressed_tokens
        return bytes(tokens).decode("utf-8")

In [15]:
with open("tests/taylorswift.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Corpus length: {len(text):,} characters")

tokenizer = BasicTokenizer()
tokenizer.train(text, vocab_size=512, verbose=False)
print(f"Trained with {len(tokenizer.lookup_table)} merges")

snippets = [
    text[:50],
    text[1000:1050],
    "Taylor Swift",
    "shake it off",
]

for snippet in snippets:
    encoded = tokenizer.encode(snippet)
    decoded = tokenizer.decode(encoded)
    ratio = len(snippet.encode("utf-8")) / len(encoded)
    print(f"\nOriginal:    {snippet!r}")
    print(f"Encoded:     {encoded}")
    print(f"Compression: {ratio:.2f}x")
    print(f"Round-trip:  {decoded == snippet}")

Corpus length: 185,561 characters
Trained with 256 merges

Original:    'Copy paste of the Wikipedia article on Taylor Swif'
Encoded:     [67, 364, 273, 112, 97, 339, 256, 445, 87, 105, 107, 105, 112, 101, 437, 338, 271, 503, 467, 279, 311]
Compression: 2.38x
Round-trip:  True

Original:    's\nRelatives\nAustin Swift (brother)\nMarjorie Finlay'
Encoded:     [115, 10, 281, 108, 388, 118, 310, 10, 65, 312, 116, 356, 379, 40, 98, 440, 267, 278, 41, 10, 453, 106, 322, 256, 70, 263, 108, 289]
Compression: 1.79x
Round-trip:  True

Original:    'Taylor Swift'
Encoded:     [365]
Compression: 12.00x
Round-trip:  True

Original:    'shake it off'
Encoded:     [115, 104, 97, 107, 256, 105, 266, 323, 102]
Compression: 1.33x
Round-trip:  True


# Step 2

Convert you `BasicTokenizer` into a `RegexTokenizer`, which takes a regex pattern and splits the text exactly as GPT-4 would. Process the parts separately as before, then concatenate the results. Retrain your tokenizer and compare the results before and after. You should see that you will now have no tokens that go across categories (numbers, letters, punctuation, more than one whitespace). Use the GPT-4 pattern:

```
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
```

In [19]:
import regex as re

In [24]:
class RegexTokenizer(BasicTokenizer):
    def __init__(self):
        super().__init__()
        GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
        self.gpt4_split_pattern = re.compile(GPT4_SPLIT_PATTERN)

    def train(self, text, vocab_size, verbose=False):
        """
        while we still need more merges:
            count pairs across ALL chunks
            find the best pair globally
            apply that merge to ALL chunks
            record the merge rule
        """
        chunks = re.findall(self.gpt4_split_pattern, text)
        tokens_per_chunk = [list(chunk.encode("utf-8")) for chunk in chunks]
        next_free_integer = 256
        while next_free_integer < vocab_size:
            occurrences = {}
            max_byte_occurrence = 0
            max_byte_tuple = None
            for tokens in tokens_per_chunk:
                for i in range(len(tokens) - 1):
                    token_tuple = (tokens[i], tokens[i + 1])
                    occurrence_count = occurrences.get(token_tuple, 0) + 1
                    occurrences[token_tuple] = occurrence_count
                    if occurrence_count > max_byte_occurrence:
                        max_byte_occurrence = occurrence_count
                        max_byte_tuple = token_tuple
            if max_byte_occurrence < 2:
                return
            self.lookup_table[max_byte_tuple] = next_free_integer
            if verbose:
                print(f"Merging {max_byte_tuple} -> {next_free_integer} (occurred {max_byte_occurrence} times)")
            condensed_tokens_across_chunks = []
            for tokens in tokens_per_chunk:
                condensed_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == max_byte_tuple:
                        condensed_tokens.append(next_free_integer)
                        i += 2
                    else:
                        condensed_tokens.append(tokens[i])
                        i += 1
                condensed_tokens_across_chunks.append(condensed_tokens)
            tokens_per_chunk = condensed_tokens_across_chunks

    def encode(self, text) -> list[int]:
        chunks = re.findall(self.gpt4_split_pattern, text)
        tokens_per_chunk = [list(chunk.encode("utf-8")) for chunk in chunks]
        encoded_tokens = list()
        for tokens in tokens_per_chunk:
            for pair, new_id in self.lookup_table.items():
                i = 0
                condensed_tokens = []
                while i < len(tokens):
                    if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                        condensed_tokens.append(new_id)
                        i += 2
                    else:
                        condensed_tokens.append(tokens[i])
                        i += 1
                tokens = condensed_tokens
            encoded_tokens.extend(tokens)
        return encoded_tokens

In [25]:
with open("tests/taylorswift.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Corpus length: {len(text):,} characters")

tokenizer = BasicTokenizer()
tokenizer.train(text, vocab_size=512, verbose=False)
print(f"Trained with {len(tokenizer.lookup_table)} merges")

snippets = [
    text[:50],
    text[1000:1050],
    "Taylor Swift",
    "shake it off",
]

for snippet in snippets:
    encoded = tokenizer.encode(snippet)
    decoded = tokenizer.decode(encoded)
    ratio = len(snippet.encode("utf-8")) / len(encoded)
    print(f"\nOriginal:    {snippet!r}")
    print(f"Encoded:     {encoded}")
    print(f"Compression: {ratio:.2f}x")
    print(f"Round-trip:  {decoded == snippet}")

Corpus length: 185,561 characters
Trained with 256 merges

Original:    'Copy paste of the Wikipedia article on Taylor Swif'
Encoded:     [67, 364, 273, 112, 97, 339, 256, 445, 87, 105, 107, 105, 112, 101, 437, 338, 271, 503, 467, 279, 311]
Compression: 2.38x
Round-trip:  True

Original:    's\nRelatives\nAustin Swift (brother)\nMarjorie Finlay'
Encoded:     [115, 10, 281, 108, 388, 118, 310, 10, 65, 312, 116, 356, 379, 40, 98, 440, 267, 278, 41, 10, 453, 106, 322, 256, 70, 263, 108, 289]
Compression: 1.79x
Round-trip:  True

Original:    'Taylor Swift'
Encoded:     [365]
Compression: 12.00x
Round-trip:  True

Original:    'shake it off'
Encoded:     [115, 104, 97, 107, 256, 105, 266, 323, 102]
Compression: 1.33x
Round-trip:  True


# Step 3

You're now ready to load the merges from the GPT-4 tokenizer and show that your tokenizer produces the identical results for both `encode` and `decode`, matching [tiktoken](https://github.com/openai/tiktoken).

```
# match this
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("hello world!!!? (안녕하세요!) lol123 😉")
text = enc.decode(ids) # get the same text back
```

Unfortunately, you will run into two issues:

1. It is not trivial to recover the raw merges from the GPT-4 tokenizer. You can easily recover what we call `vocab` here, and what they call and store under `enc._mergeable_ranks`. Feel free to copy paste the `recover_merges` function in [`minbpe/gpt4.py`](https://github.com/karpathy/minbpe/blob/master/minbpe/gpt4.py), which takes these ranks and returns the raw merges. If you wish to know how this function works, read [this](https://github.com/openai/tiktoken/issues/60) and [this](https://github.com/karpathy/minbpe/issues/11#issuecomment-1950805306). Basically, under some conditions it is enough to only store the parent nodes (and their rank) and get rid of the precise details of which children merged up to any parent.
2. Second, the GPT-4 tokenizer for some reason permutes its raw bytes. It stores this permutation in the first 256 elements of the mergeable ranks, so you can recover this byte shuffle relatively simply as `byte_shuffle = {i: enc._mergeable_ranks[bytes([i])] for i in range(256)}`. In both your encode and decode, you'll have to shuffle bytes around accordingly. If you're stuck, reference the `minbpe/gpt4.py` file for hints.

### Code from [here](https://github.com/karpathy/minbpe/blob/1acefe89412b20245db5a22d2a02001e547dc602/minbpe/gpt4.py#L11-L47)

In [27]:
import tiktoken

def bpe(mergeable_ranks, token, max_rank):
    # helper function used in get_gpt4_merges() to reconstruct the merge forest
    parts = [bytes([b]) for b in token]
    while True:
        min_idx = None
        min_rank = None
        for i, pair in enumerate(zip(parts[:-1], parts[1:])):
            rank = mergeable_ranks.get(pair[0] + pair[1])
            if rank is not None and (min_rank is None or rank < min_rank):
                min_idx = i
                min_rank = rank
        if min_rank is None or (max_rank is not None and min_rank >= max_rank):
            break
        assert min_idx is not None
        parts = parts[:min_idx] + [parts[min_idx] + parts[min_idx + 1]] + parts[min_idx + 2:]
    return parts


def recover_merges(mergeable_ranks):
    # the `merges` are already the byte sequences in their merged state.
    # so we have to recover the original pairings. We can do this by doing
    # a small BPE training run on all the tokens, in their order.
    # also see https://github.com/openai/tiktoken/issues/60
    # also see https://github.com/karpathy/minbpe/issues/11#issuecomment-1950805306
    merges = {}
    for token, rank in mergeable_ranks.items():
        if len(token) == 1:
            continue # skip raw bytes
        pair = tuple(bpe(mergeable_ranks, token, max_rank=rank))
        assert len(pair) == 2
        # recover the integer ranks of the pair
        ix0 = mergeable_ranks[pair[0]]
        ix1 = mergeable_ranks[pair[1]]
        merges[(ix0, ix1)] = rank

    return merges

In [35]:
enc = tiktoken.get_encoding("cl100k_base")
mergeable_ranks = enc._mergeable_ranks
# the merges are those of gpt4, but we have to recover them
merges = recover_merges(mergeable_ranks)
merges

{(220, 220): 256,
 (256, 256): 257,
 (72, 77): 258,
 (220, 83): 259,
 (257, 257): 260,
 (68, 81): 261,
 (256, 220): 262,
 (78, 77): 263,
 (220, 64): 264,
 (81, 68): 265,
 (64, 83): 266,
 (82, 83): 267,
 (68, 77): 268,
 (78, 81): 269,
 (259, 71): 270,
 (198, 198): 271,
 (220, 66): 272,
 (75, 68): 273,
 (220, 82): 274,
 (72, 83): 275,
 (64, 77): 276,
 (64, 81): 277,
 (64, 75): 278,
 (270, 68): 279,
 (26, 198): 280,
 (220, 79): 281,
 (220, 69): 282,
 (78, 84): 283,
 (220, 28): 284,
 (72, 82): 285,
 (257, 262): 286,
 (258, 70): 287,
 (68, 82): 288,
 (220, 86): 289,
 (72, 263): 290,
 (68, 67): 291,
 (72, 66): 292,
 (220, 65): 293,
 (220, 67): 294,
 (68, 83): 295,
 (220, 76): 296,
 (220, 78): 297,
 (197, 197): 298,
 (81, 78): 299,
 (64, 82): 300,
 (68, 75): 301,
 (66, 83): 302,
 (77, 67): 303,
 (220, 258): 304,
 (220, 71): 305,
 (268, 83): 306,
 (72, 67): 307,
 (220, 77): 308,
 (64, 76): 309,
 (260, 262): 310,
 (259, 78): 311,
 (220, 265): 312,
 (12, 12): 313,
 (220, 90): 314,
 (297, 69): 31

In [38]:
byte_shuffle = {i: enc._mergeable_ranks[bytes([i])] for i in range(256)}
byte_shuffle

{0: 188,
 1: 189,
 2: 190,
 3: 191,
 4: 192,
 5: 193,
 6: 194,
 7: 195,
 8: 196,
 9: 197,
 10: 198,
 11: 199,
 12: 200,
 13: 201,
 14: 202,
 15: 203,
 16: 204,
 17: 205,
 18: 206,
 19: 207,
 20: 208,
 21: 209,
 22: 210,
 23: 211,
 24: 212,
 25: 213,
 26: 214,
 27: 215,
 28: 216,
 29: 217,
 30: 218,
 31: 219,
 32: 220,
 33: 0,
 34: 1,
 35: 2,
 36: 3,
 37: 4,
 38: 5,
 39: 6,
 40: 7,
 41: 8,
 42: 9,
 43: 10,
 44: 11,
 45: 12,
 46: 13,
 47: 14,
 48: 15,
 49: 16,
 50: 17,
 51: 18,
 52: 19,
 53: 20,
 54: 21,
 55: 22,
 56: 23,
 57: 24,
 58: 25,
 59: 26,
 60: 27,
 61: 28,
 62: 29,
 63: 30,
 64: 31,
 65: 32,
 66: 33,
 67: 34,
 68: 35,
 69: 36,
 70: 37,
 71: 38,
 72: 39,
 73: 40,
 74: 41,
 75: 42,
 76: 43,
 77: 44,
 78: 45,
 79: 46,
 80: 47,
 81: 48,
 82: 49,
 83: 50,
 84: 51,
 85: 52,
 86: 53,
 87: 54,
 88: 55,
 89: 56,
 90: 57,
 91: 58,
 92: 59,
 93: 60,
 94: 61,
 95: 62,
 96: 63,
 97: 64,
 98: 65,
 99: 66,
 100: 67,
 101: 68,
 102: 69,
 103: 70,
 104: 71,
 105: 72,
 106: 73,
 107: 74,
 108: 7

### Amended tokenizer

In [47]:
class GPT4Tokenizer(RegexTokenizer):
    def __init__(self):
        super().__init__()
        self.lookup_table = merges
        self.byte_shuffle = byte_shuffle
        self.byte_shuffle_inverted = {v:k for (k,v) in byte_shuffle.items()}

    def encode(self, text) -> list[int]:
        chunks = re.findall(self.gpt4_split_pattern, text)
        tokens_per_chunk = [list(chunk.encode("utf-8")) for chunk in chunks]
        encoded_tokens = list()
        for tokens in tokens_per_chunk:
            # Accounting for GPT 4's random byte shuffle
            tokens = [self.byte_shuffle[token] for token in tokens]
            for pair, new_id in self.lookup_table.items():
                i = 0
                condensed_tokens = []
                while i < len(tokens):
                    if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                        condensed_tokens.append(new_id)
                        i += 2
                    else:
                        condensed_tokens.append(tokens[i])
                        i += 1
                tokens = condensed_tokens
            encoded_tokens.extend(tokens)
        return encoded_tokens

    def decode(self, tokens: list[int]) -> str:
        for pair, new_id in reversed(self.lookup_table.items()):
            decompressed_tokens = []
            for i in range(0, len(tokens)):
                if tokens[i] == new_id:
                    decompressed_tokens.append(pair[0])
                    decompressed_tokens.append(pair[1])
                else:
                    decompressed_tokens.append(tokens[i])
            tokens = decompressed_tokens
        tokens = [self.byte_shuffle_inverted[token] for token in tokens]
        return bytes(tokens).decode("utf-8")

### Answer

In [50]:
# Theirs
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
ids = enc.encode("hello world!!!? (안녕하세요!) lol123 😉")
text = enc.decode(ids)
text

'hello world!!!? (안녕하세요!) lol123 😉'

In [53]:
# Ours
tokenizer = GPT4Tokenizer()
our_ids = tokenizer.encode("hello world!!!? (안녕하세요!) lol123 😉")
assert our_ids == ids
our_text = tokenizer.decode(ids)
our_text

'hello world!!!? (안녕하세요!) lol123 😉'

# Step 4

(Optional, irritating, not obviously useful) Add the ability to handle special tokens. You'll then be able to match the output of tiktoken even when special tokens are present, e.g.:

```
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("<|endoftext|>hello world", allowed_special="all")
```

Without `allowed_special` tiktoken will error.

In [75]:
import regex as re
special_tokens = ["<|endoftext|>", "<|fim_prefix|>"]
pattern = "(" + "|".join(re.escape(t) for t in special_tokens) + ")"
text = "hello<|endoftext|>world<|fim_prefix|>foo"
print(re.split(pattern, text))

['hello', '<|endoftext|>', 'world', '<|fim_prefix|>', 'foo']


In [77]:
special_tokens = enc._special_tokens
special_tokens

{'<|endoftext|>': 100257,
 '<|fim_prefix|>': 100258,
 '<|fim_middle|>': 100259,
 '<|fim_suffix|>': 100260,
 '<|endofprompt|>': 100276}

### Amended tokenizer

In [91]:
class GPT4Tokenizer(RegexTokenizer):
    def __init__(self):
        super().__init__()
        self.lookup_table = merges
        self.byte_shuffle = byte_shuffle
        self.byte_shuffle_inverted = {v:k for (k,v) in byte_shuffle.items()}
        self.special_tokens = special_tokens
        self.special_tokens_inverted = {v:k for (k,v) in special_tokens.items()}
        self.special_tokens_pattern = "(" + "|".join(re.escape(t) for t in special_tokens) + ")"

    def encode(self, text) -> list[int]:
        # Special tokens preprocessing
        tokens_per_chunk = list()
        for text in re.split(self.special_tokens_pattern, text):
            if text == '':
                continue
            elif text in self.special_tokens:
                tokens_per_chunk.append(self.special_tokens[text])
            else:
                chunks = re.findall(self.gpt4_split_pattern, text)
                chunks = [list(chunk.encode("utf-8")) for chunk in chunks]
                tokens_per_chunk.extend(chunks)

        encoded_tokens = list()
        for tokens in tokens_per_chunk:
            if isinstance(tokens, int):
                encoded_tokens.append(tokens)
            else:
                # Accounting for GPT 4's random byte shuffle
                tokens = [self.byte_shuffle[token] for token in tokens]
                for pair, new_id in self.lookup_table.items():
                    i = 0
                    condensed_tokens = []
                    while i < len(tokens):
                        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                            condensed_tokens.append(new_id)
                            i += 2
                        else:
                            condensed_tokens.append(tokens[i])
                            i += 1
                    tokens = condensed_tokens
                encoded_tokens.extend(tokens)
        return encoded_tokens

    def decode(self, tokens: list[int]) -> str:
        for pair, new_id in reversed(self.lookup_table.items()):
            decompressed_tokens = []
            for i in range(0, len(tokens)):
                if tokens[i] == new_id:
                    decompressed_tokens.append(pair[0])
                    decompressed_tokens.append(pair[1])
                else:
                    decompressed_tokens.append(tokens[i])
            tokens = decompressed_tokens
        tokens = [self.byte_shuffle_inverted[token] if token not in self.special_tokens_inverted else token for token in tokens]
        # Handling for special tokens
        decoded = list()
        i = 0
        while i < len(tokens):
            j = i
            accumulated_bytes = list()
            while j < len(tokens) and tokens[j] not in self.special_tokens_inverted:
                accumulated_bytes.append(tokens[j])
                j += 1
            decoded.append(bytes(accumulated_bytes).decode("utf-8"))
            if j < len(tokens):
                decoded.append(self.special_tokens_inverted[tokens[j]])
                j += 1
            i = j
        return "".join(decoded)

### Answer

In [93]:
# Theirs
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("<|endoftext|>hello world", allowed_special="all")
text = enc.decode(ids)
text

'<|endoftext|>hello world'

In [95]:
# Ours
tokenizer = GPT4Tokenizer()
our_ids = tokenizer.encode("<|endoftext|>hello world")
assert our_ids == ids
our_text = tokenizer.decode(ids)
our_text

'<|endoftext|>hello world'

# Step 5

If you've made it this far, you're now a pro at LLM Tokenization! Sadly, you're not exactly done yet because a lot of LLMs outside of OpenAI (e.g. Llama, Mistral) use [sentencepiece](https://github.com/google/sentencepiece) instead. Primary difference being that sentencepiece runs BPE directly on Unicode code points instead of on UTF-8 encoded bytes. Feel free to explore sentencepiece on your own (good luck, it's not too pretty), and stretch goal if you really experience and suffer from the burden of time, re-write your BPE to be on Unicode code points and match the Llama 2 tokenizer.